# Notebook enrich dataset 100 căn

Notebook này phục vụ bước làm giàu dữ liệu cho bộ **100 căn** của member-3.

Mục tiêu:
- đọc file `data/go_vap_tan_binh_100.json`
- khai báo POI tham chiếu cho `Gò Vấp` và `Tân Bình`
- tính khoảng cách đến POI gần nhất cho từng căn
- bổ sung các trường `nearest_*_name` và `near_*_count_1km`
- xuất ra file `data/go_vap_tan_binh_100_enriched.json`

Mỗi cell bên dưới đều ghi rõ đang làm bước gì để sau này chỉ cần chạy tuần tự.

## Bước 1. Import thư viện và khai báo đường dẫn

Cell này chuẩn bị:
- các thư viện chuẩn của Python
- hàm tự dò thư mục gốc của project
- đường dẫn file input và output

In [ ]:
import json
import math
from pathlib import Path


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / 'data' / 'go_vap_tan_binh_100.json').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Không tìm thấy thư mục gốc của project chứa data/go_vap_tan_binh_100.json')


ROOT = find_project_root(Path.cwd())
INPUT_JSON = ROOT / 'data' / 'go_vap_tan_binh_100.json'
OUTPUT_JSON = ROOT / 'data' / 'go_vap_tan_binh_100_enriched.json'

print('Project root:', ROOT)
print('Input JSON:', INPUT_JSON)
print('Output JSON:', OUTPUT_JSON)


## Bước 2. Khai báo POI tham chiếu cho từng quận

Cell này chứa các điểm tham chiếu để tính khoảng cách.

Mỗi quận có 5 nhóm POI:
- `school`
- `park`
- `hospital`
- `supermarket`
- `boulevard`


In [ ]:
DISTRICT_POIS = {
    'Gò Vấp': {
        'school': [
            {'name': 'Trường THCS Nguyễn Du', 'lat': 10.8405, 'lon': 106.6550},
            {'name': 'Trường THPT Trần Hưng Đạo', 'lat': 10.8370, 'lon': 106.6635},
            {'name': 'Trường TH Lương Thế Vinh', 'lat': 10.8480, 'lon': 106.6520},
            {'name': 'Trường THCS Gò Vấp', 'lat': 10.8330, 'lon': 106.6450},
            {'name': 'Trường THPT Nguyễn Công Trứ', 'lat': 10.8450, 'lon': 106.6700},
        ],
        'park': [
            {'name': 'Công viên Gia Định', 'lat': 10.8190, 'lon': 106.6770},
            {'name': 'Công viên Phần mềm Quang Trung', 'lat': 10.8460, 'lon': 106.6330},
            {'name': 'Công viên phường 12', 'lat': 10.8350, 'lon': 106.6400},
            {'name': 'Công viên Làng Hoa', 'lat': 10.8430, 'lon': 106.6580},
        ],
        'hospital': [
            {'name': 'Bệnh viện Quận Gò Vấp', 'lat': 10.8380, 'lon': 106.6500},
            {'name': 'Bệnh viện 175', 'lat': 10.8570, 'lon': 106.6640},
            {'name': 'Phòng khám Đa khoa Sài Gòn', 'lat': 10.8450, 'lon': 106.6620},
        ],
        'supermarket': [
            {'name': 'Emart Gò Vấp', 'lat': 10.8345, 'lon': 106.6575},
            {'name': 'Co.opmart Quang Trung', 'lat': 10.8400, 'lon': 106.6470},
            {'name': 'VinMart Thống Nhất', 'lat': 10.8430, 'lon': 106.6650},
            {'name': 'Mega Market Quang Trung', 'lat': 10.8310, 'lon': 106.6390},
        ],
        'boulevard': [
            {'name': 'Quang Trung', 'lat': 10.8380, 'lon': 106.6450},
            {'name': 'Nguyễn Oanh', 'lat': 10.8420, 'lon': 106.6700},
            {'name': 'Phạm Văn Đồng', 'lat': 10.8200, 'lon': 106.6830},
            {'name': 'Lê Đức Thọ', 'lat': 10.8530, 'lon': 106.6580},
        ],
    },
    'Tân Bình': {
        'school': [
            {'name': 'Trường THPT Nguyễn Thượng Hiền', 'lat': 10.7998, 'lon': 106.6524},
            {'name': 'Trường THCS Hoàng Hoa Thám', 'lat': 10.8038, 'lon': 106.6402},
            {'name': 'Trường TH Tân Sơn', 'lat': 10.8123, 'lon': 106.6420},
            {'name': 'Trường THCS Ngô Sĩ Liên', 'lat': 10.7928, 'lon': 106.6507},
            {'name': 'Trường THPT Tân Bình', 'lat': 10.7903, 'lon': 106.6466},
        ],
        'park': [
            {'name': 'Công viên Hoàng Văn Thụ', 'lat': 10.8019, 'lon': 106.6648},
            {'name': 'Công viên Gia Định', 'lat': 10.8190, 'lon': 106.6770},
            {'name': 'Công viên Bàu Cát', 'lat': 10.7936, 'lon': 106.6396},
            {'name': 'Công viên Tân Phước', 'lat': 10.7898, 'lon': 106.6535},
        ],
        'hospital': [
            {'name': 'Bệnh viện Thống Nhất', 'lat': 10.8011, 'lon': 106.6478},
            {'name': 'Bệnh viện Tân Bình', 'lat': 10.7947, 'lon': 106.6527},
            {'name': 'Bệnh viện Phụ sản Mê Kông', 'lat': 10.8011, 'lon': 106.6566},
            {'name': 'Bệnh viện Quốc tế CityCare Tân Bình', 'lat': 10.8065, 'lon': 106.6388},
        ],
        'supermarket': [
            {'name': 'GO! Trường Chinh', 'lat': 10.8068, 'lon': 106.6248},
            {'name': 'Lotte Mart Cộng Hòa', 'lat': 10.8018, 'lon': 106.6446},
            {'name': 'Co.opmart Hoàng Văn Thụ', 'lat': 10.7994, 'lon': 106.6622},
            {'name': 'VinMart Cộng Hòa', 'lat': 10.8026, 'lon': 106.6510},
        ],
        'boulevard': [
            {'name': 'Cộng Hòa', 'lat': 10.8010, 'lon': 106.6485},
            {'name': 'Trường Chinh', 'lat': 10.8060, 'lon': 106.6330},
            {'name': 'Hoàng Văn Thụ', 'lat': 10.7996, 'lon': 106.6640},
            {'name': 'Lý Thường Kiệt', 'lat': 10.7889, 'lon': 106.6520},
        ],
    },
}

list(DISTRICT_POIS.keys())


## Bước 3. Viết các hàm tính khoảng cách và enrich từng căn

Cell này định nghĩa:
- công thức `haversine` để tính khoảng cách theo mét
- hàm enrich một căn bất động sản bằng POI của đúng quận

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    earth_radius_m = 6371000
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return earth_radius_m * c


def enrich_property(prop):
    district = prop['district']
    if district not in DISTRICT_POIS:
        raise ValueError(f'Không hỗ trợ enrich cho quận: {district}')

    lat = prop['latitude']
    lon = prop['longitude']
    pois_by_type = DISTRICT_POIS[district]
    enriched = dict(prop)

    for poi_type, pois in pois_by_type.items():
        distances = [haversine_m(lat, lon, poi['lat'], poi['lon']) for poi in pois]
        nearest_idx = distances.index(min(distances))
        enriched[f'distance_to_nearest_{poi_type}_m'] = round(min(distances))
        enriched[f'nearest_{poi_type}_name'] = pois[nearest_idx]['name']
        enriched[f'near_{poi_type}_count_1km'] = sum(1 for distance in distances if distance <= 1000)

    return enriched


## Bước 4. Đọc file clean dataset và enrich toàn bộ 100 căn

Cell này:
- đọc file `go_vap_tan_binh_100.json`
- enrich từng căn
- in thử một mẫu để kiểm tra nhanh

In [ ]:
with INPUT_JSON.open('r', encoding='utf-8') as f:
    properties = json.load(f)

enriched_properties = [enrich_property(prop) for prop in properties]

print('Tổng số căn:', len(enriched_properties))
print('Một số quận có trong dữ liệu:', sorted({item['district'] for item in enriched_properties}))
enriched_properties[0]


## Bước 5. Kiểm tra nhanh các cột mới sau enrich

Cell này giúp xác nhận:
- các cột `distance_to_nearest_*` đã có giá trị
- các trường `nearest_*_name` và `near_*_count_1km` đã được thêm

In [ ]:
sample = enriched_properties[0]
check_keys = [
    'distance_to_nearest_school_m', 'nearest_school_name', 'near_school_count_1km',
    'distance_to_nearest_park_m', 'nearest_park_name', 'near_park_count_1km',
    'distance_to_nearest_hospital_m', 'nearest_hospital_name', 'near_hospital_count_1km',
    'distance_to_nearest_supermarket_m', 'nearest_supermarket_name', 'near_supermarket_count_1km',
    'distance_to_nearest_boulevard_m', 'nearest_boulevard_name', 'near_boulevard_count_1km',
]

{key: sample[key] for key in check_keys}


## Bước 6. Ghi file enriched ra thư mục data/

Cell này xuất kết quả cuối cùng thành file `go_vap_tan_binh_100_enriched.json`.

In [ ]:
with OUTPUT_JSON.open('w', encoding='utf-8') as f:
    json.dump(enriched_properties, f, ensure_ascii=False, indent=2)

print('Đã ghi file:', OUTPUT_JSON)
print('Kích thước file:', OUTPUT_JSON.stat().st_size, 'bytes')


## Bước 7. Tổng kết nhanh sau khi enrich

Cell này in thống kê cuối để xác nhận:
- đủ 100 căn
- đủ 50 căn Gò Vấp và 50 căn Tân Bình
- các cột khoảng cách không còn `None`

In [ ]:
district_counts = {}
for item in enriched_properties:
    district_counts[item['district']] = district_counts.get(item['district'], 0) + 1

null_distance_count = sum(
    1
    for item in enriched_properties
    for key, value in item.items()
    if key.startswith('distance_to_nearest_') and value is None
)

print('Phân bố theo quận:', district_counts)
print('Số giá trị distance còn None:', null_distance_count)
